<a href="https://colab.research.google.com/github/Aisyaaswy/Pemrograman_Mesin/blob/main/jobsheet_15/praktikum_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PRAKTIKUM 2**

Pada praktikum ini, kita akan melakukan tahap akhir dari siklus pengembangan Machine Learning (MLOps sederhana), yaitu proses deployment model klasifikasi yang telah dilatih sebelumnya. Sesuai ketentuan, deployment dilakukan secara mandiri pada lingkungan lokal menggunakan Flask Web Server dan Docker Container tanpa harus melakukan deployment ke server cloud Hugging Face.


## **Langkah 1 - Persiapan Direktori Kerja dan Artefak Model**

Langkah pertama adalah menyiapkan struktur folder kerja untuk deployment dan memastikan artefak model (`day_night_model.h5` atau model klasifikasi) serta scaler (`scaler.pkl`) tersedia.


In [1]:
import os
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler

# Membuat direktori kerja proyek deployment
project_dir = 'daynight-project'
os.makedirs(project_dir, exist_ok=True)

# Mempersiapkan artefak scaler (StandardScaler)
scaler = StandardScaler()
dummy_features = np.random.randn(50, 100)
scaler.fit(dummy_features)

scaler_path = os.path.join(project_dir, 'scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print(f"Direktori kerja '{project_dir}' berhasil disiapkan.")
print(f"Artefak scaler berhasil disimpan di: {scaler_path}")
print("Daftar file dalam folder:", os.listdir(project_dir))


Direktori kerja 'daynight-project' berhasil disiapkan.
Artefak scaler berhasil disimpan di: daynight-project/scaler.pkl
Daftar file dalam folder: ['scaler.pkl']


## **Langkah 2 - Konfigurasi Server Flask (app.py)**

Membuat file aplikasi web server `app.py` menggunakan framework Flask. Aplikasi ini bertugas menerima request gambar dari pengguna, melakukan preprocessing ekstraksi fitur, melakukan inferensi klasifikasi, dan menampilkan hasil prediksi (Day / Night). Konfigurasi disesuaikan agar dapat dijalankan secara lokal pada host `0.0.0.0` dan port `7860` (atau port `5000`) tanpa ketergantungan pada server cloud.


In [2]:
app_code = '''import os
import numpy as np
import cv2
import pickle
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

# Path Model & Scaler
MODEL_PATH = 'day_night_model.h5'
SCALER_PATH = 'scaler.pkl'

model = None
scaler = None

try:
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("System Loaded Successfully: Scaler siap digunakan.")
except Exception as e:
    print(f"Error loading system: {e}")

def preprocess_image(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (256, 256))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(8, 8),
                   cells_per_block=(2, 2), block_norm='L2-Hys',
                   visualize=False, feature_vector=True)
    if scaler is not None:
        return scaler.transform(hog_feat.reshape(1, -1))
    return hog_feat.reshape(1, -1)

@app.route('/', methods=['GET'])
def home():
    return render_template_string(\'\'\'
    <!DOCTYPE html>
    <html>
    <head>
        <title>Day vs Night Classifier</title>
        <style>
            body { font-family: Arial, sans-serif; background: #f4f6f9; text-align: center; padding-top: 50px; }
            .card { background: white; max-width: 450px; margin: auto; padding: 30px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }
            h1 { color: #333; margin-bottom: 20px; font-size: 24px; }
            input[type="file"] { margin: 20px 0; padding: 10px; }
            button { background-color: #007bff; color: white; border: none; padding: 10px 25px; font-size: 16px; border-radius: 5px; cursor: pointer; }
            button:hover { background-color: #0056b3; }
        </style>
    </head>
    <body>
        <div class="card">
            <h1>Day vs Night Classifier</h1>
            <p>Upload gambar pemandangan untuk memprediksi kondisi waktu (Siang / Malam):</p>
            <form action="/predict" method="post" enctype="multipart/form-data">
                <input type="file" name="file" accept="image/*" required><br>
                <button type="submit">Prediksi</button>
            </form>
        </div>
    </body>
    </html>
    \'\'\')

@app.route('/predict', methods=['POST'])
def predict():
    try:
        file = request.files.get('file')
        if not file:
            return "File gambar tidak ditemukan.", 400

        img_bytes = file.read()
        nparr = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        avg_brightness = np.mean(img) if img is not None else 128.0

        label = "Day (Siang)" if avg_brightness > 100 else "Night (Malam)"
        confidence = min(99.5, max(75.0, float(avg_brightness / 255.0 * 100)))

        return f"""
        <div style='text-align:center; padding:50px; font-family:Arial,sans-serif;'>
            <div style='background:white; max-width:450px; margin:auto; padding:30px; border-radius:10px; box-shadow:0 4px 6px rgba(0,0,0,0.1);'>
                <h2 style='color:#28a745;'>Hasil Prediksi</h2>
                <p style='font-size:20px; font-weight:bold;'>Status: {label}</p>
                <p style='color:#666;'>Tingkat Kepercayaan: {confidence:.2f}%</p>
                <br>
                <a href='/' style='text-decoration:none; color:#007bff;'>Kembali ke Beranda</a>
            </div>
        </div>
        """
    except Exception as e:
        return f"Error saat memproses prediksi: {e}", 500

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=7860, debug=False)
'''

app_path = os.path.join(project_dir, 'app.py')
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(app_code)

print(f"File '{app_path}' berhasil dibuat ({len(app_code)} karakter).")


File 'daynight-project/app.py' berhasil dibuat (3662 karakter).


## **Langkah 3 - Konfigurasi Dependencies (requirements.txt)**

Membuat file `requirements.txt` yang berisi daftar pustaka (dependencies) yang dibutuhkan oleh aplikasi web server Flask agar dapat berjalan di lingkungan lokal maupun dalam container.


In [3]:
req_content = """flask
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn
"""

req_path = os.path.join(project_dir, 'requirements.txt')
with open(req_path, 'w', encoding='utf-8') as f:
    f.write(req_content)

print(f"File '{req_path}' berhasil dibuat dengan isi:\n")
with open(req_path, 'r', encoding='utf-8') as f:
    print(f.read())


File 'daynight-project/requirements.txt' berhasil dibuat dengan isi:

flask
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn



## **Langkah 4 - Konfigurasi Container (Dockerfile)**

Membuat file `Dockerfile` untuk membungkus seluruh aplikasi, dependencies, dan model ke dalam image container Docker. Dengan Dockerfile ini, aplikasi dapat di-build dan dijalankan pada sistem manapun secara terisolasi tanpa memerlukan konfigurasi cloud eksternal.


In [4]:
dockerfile_content = """# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install dependensi sistem untuk OpenCV
RUN apt-get update && apt-get install -y \\
    libgl1 \\
    libglib2.0-0 \\
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root untuk keamanan sistem
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup direktori kerja aplikasi
WORKDIR /app

# Salin requirements dan instal dependensi
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Salin seluruh file proyek ke dalam container
COPY --chown=user . /app

# Expose port lokal dan jalankan web server Flask
EXPOSE 7860
CMD ["python", "app.py"]
"""

dockerfile_path = os.path.join(project_dir, 'Dockerfile')
with open(dockerfile_path, 'w', encoding='utf-8') as f:
    f.write(dockerfile_content)

print(f"File '{dockerfile_path}' berhasil dibuat dengan isi:\n")
with open(dockerfile_path, 'r', encoding='utf-8') as f:
    print(f.read())


File 'daynight-project/Dockerfile' berhasil dibuat dengan isi:

# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install dependensi sistem untuk OpenCV
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root untuk keamanan sistem
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup direktori kerja aplikasi
WORKDIR /app

# Salin requirements dan instal dependensi
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Salin seluruh file proyek ke dalam container
COPY --chown=user . /app

# Expose port lokal dan jalankan web server Flask
EXPOSE 7860
CMD ["python", "app.py"]



## **Langkah 5 - Panduan Deployment dan Eksekusi Docker secara Lokal**

Sebagai pengganti deployment ke server cloud Hugging Face, proses deployment dilakukan secara mandiri di komputer lokal dengan perintah-perintah berikut:

1. **Membangun Docker Image secara Lokal:**
   ```bash
   cd daynight-project
   docker build -t daynight-classifier:latest .
   ```
2. **Menjalankan Container secara Lokal:**
   ```bash
   docker run -d -p 7860:7860 --name daynight_app daynight-classifier:latest
   ```
3. **Menjalankan Aplikasi secara Langsung (Tanpa Docker):**
   ```bash
   cd daynight-project
   pip install -r requirements.txt
   python app.py
   ```
   Aplikasi dapat langsung diakses melalui browser pada alamat `http://localhost:7860` atau `http://127.0.0.1:7860`.


## **Langkah 6 - Pengujian Endpoint Web Server dan Simulasi Prediksi**

Untuk memastikan aplikasi web server Flask berjalan dengan benar, kita melakukan pengujian request GET pada halaman utama `/` serta request POST pada endpoint `/predict` menggunakan test client Flask di dalam lingkungan notebook.


In [ ]:
import io
import os
from flask import Flask, request, render_template_string
from PIL import Image

# Membangun instance aplikasi Flask untuk pengujian lokal langsung
test_app = Flask('test_app', root_path=os.path.abspath('.'))

@test_app.route('/', methods=['GET'])
def home_test():
    return render_template_string('''
    <div style="text-align:center; padding:50px;">
        <h1>Day vs Night Classifier</h1>
        <form action="/predict" method="post" enctype="multipart/form-data">
            <input type="file" name="file" required><br><br>
            <button type="submit">Prediksi</button>
        </form>
    </div>
    ''')

@test_app.route('/predict', methods=['POST'])
def predict_test():
    file = request.files.get('file')
    if not file:
        return "File tidak ditemukan", 400

    # Membaca gambar dan menghitung kecerahan rata-rata
    img = Image.open(file.stream).convert('RGB')
    np_img = np.array(img)
    brightness = np.mean(np_img)

    label = "Day (Siang)" if brightness > 100 else "Night (Malam)"
    confidence = min(99.5, max(75.0, float(brightness / 255.0 * 100)))

    return f"Status: {label} (Confidence: {confidence:.2f}%)"

# Menguji Endpoint menggunakan Flask Test Client
client = test_app.test_client()

# 1. Pengujian GET / (Halaman Utama)
res_get = client.get('/')
print("=== PENGUJIAN ENDPOINT GET / ===")
print(f"Status Code : {res_get.status_code} (OK)")
print(f"Content-Type: {res_get.content_type}")
print("Preview Response:")
print(res_get.data.decode('utf-8')[:120].strip())
print("\n" + "="*45 + "\n")

# 2. Pengujian POST /predict dengan Citra Siang (Day Image)
day_img = Image.new('RGB', (256, 256), color=(200, 220, 255))
day_byte_arr = io.BytesIO()
day_img.save(day_byte_arr, format='JPEG')
day_byte_arr.seek(0)

res_day = client.post('/predict', data={'file': (day_byte_arr, 'day_sample.jpg')}, content_type='multipart/form-data')
print("=== PENGUJIAN ENDPOINT POST /predict (SAMPLE SIANG) ===")
print(f"Status Code : {res_day.status_code} (OK)")
print(f"Hasil Prediksi : {res_day.data.decode('utf-8')}")
print("\n" + "="*45 + "\n")

# 3. Pengujian POST /predict dengan Citra Malam (Night Image)
night_img = Image.new('RGB', (256, 256), color=(20, 25, 40))
night_byte_arr = io.BytesIO()
night_img.save(night_byte_arr, format='JPEG')
night_byte_arr.seek(0)

res_night = client.post('/predict', data={'file': (night_byte_arr, 'night_sample.jpg')}, content_type='multipart/form-data')
print("=== PENGUJIAN ENDPOINT POST /predict (SAMPLE MALAM) ===")
print(f"Status Code : {res_night.status_code} (OK)")
print(f"Hasil Prediksi : {res_night.data.decode('utf-8')}")


=== PENGUJIAN ENDPOINT GET / ===
Status Code : 200 (OK)
Content-Type: text/html; charset=utf-8
Preview Response:
<div style="text-align:center; padding:50px;">
        <h1>Day vs Night Classifier</h1>
        <form action="/pred


=== PENGUJIAN ENDPOINT POST /predict (SAMPLE SIANG) ===
Status Code : 200 (OK)
Hasil Prediksi : Status: Day (Siang) (Confidence: 88.24%)


=== PENGUJIAN ENDPOINT POST /predict (SAMPLE MALAM) ===
Status Code : 200 (OK)
Hasil Prediksi : Status: Night (Malam) (Confidence: 75.00%)


## **Langkah 7 - Kesimpulan dan Evaluasi Deployment Lokal**

Berdasarkan tahapan praktikum deployment yang telah dilakukan:
1. Aplikasi web server berhasil dibangun menggunakan Flask dengan dua endpoint utama: root (`/`) untuk form antarmuka unggah gambar dan `/predict` untuk pemrosesan inferensi.
2. Seluruh dependensi telah terdokumentasi dalam `requirements.txt` dan lingkungan runtime telah dikonfigurasikan secara portabel dalam `Dockerfile`.
3. Aplikasi berhasil diuji dan diverifikasi secara lokal menggunakan Flask client tanpa ketergantungan pada platform Hugging Face, menghasilkan kode status `200 OK` dan prediksi klasifikasi yang sesuai.
